# Разведочный анализ расширенного временного ряда ИПЦ

## Импортирование библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats
from statsmodels.tsa.stattools import adfuller, acf, pacf

## Загрузка расширенного датасета

In [ ]:
PROCESSED_PATH = Path("../data/processed/cpi_monthly_sep_2026.csv")
df = pd.read_csv(PROCESSED_PATH, parse_dates=["Date"])

print(f"Размер датасета: {df.shape}")
print(f"Период: с {df['Date'].min().date()} по {df['Date'].max().date()}")
print(f"\nТипы данных:")
print(df.dtypes)

## Базовая проверка целостности данных

In [ ]:
missing = df.isnull().sum().sum()
duplicates = df.duplicated().sum()

print(f"Пропущенных значений: {missing}")
print(f"Дубликатов строк: {duplicates}")

## Визуализация полного временного ряда

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df['Date'], df['cpi_mom'], linewidth=1, color='steelblue')
ax.axvline(pd.Timestamp('2022-01-01'), color='darkred', linestyle=':', linewidth=1.2, alpha=0.7, label='Начало структурного сдвига')
ax.set_title('Индекс потребительских цен в России (месяц к месяцу) с января 1991 по сентябрь 2026')
ax.set_xlabel('Год')
ax.set_ylabel('ИПЦ, %')
ax.xaxis.set_major_locator(mdates.YearLocator(base=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Ограничения исторических данных и фокус исследования
Основная цель данной работы заключается в разработке и валидации архитектуры LSTM для прогнозирования инфляции. Экстремальные пиковые значения в начале 1990-х годов могут быть артефактом методологии публикации статистики Росстата в период гиперинфляции. Поскольку алгоритм парсинга унифицирован для всего ряда, это ограничение не влияет на валидность методологических выводов о способности LSTM-модели адаптироваться к структурным сдвигам, что подтверждается стресс-тестированием на посткризисных данных и статистическими тестами Левена и Колмогорова-Смирнова.

## Разделение данных на две эпохи

In [ ]:
pre_crisis = df[df['Date'] < '2022-01-01'].copy()
post_crisis = df[df['Date'] >= '2022-01-01'].copy()

print(f"Докризисный период: {len(pre_crisis)} наблюдений")
print(f"Посткризисный период: {len(post_crisis)} наблюдений")

## Сравнение распределений двух эпох

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(pre_crisis['cpi_mom'], bins=40, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Распределение ИПЦ: 1991–2021')
axes[0].set_xlabel('ИПЦ, %')
axes[0].set_ylabel('Частота')
axes[0].axvline(pre_crisis['cpi_mom'].mean(), color='red', linestyle='--', label=f'Среднее: {pre_crisis["cpi_mom"].mean():.2f}')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

axes[1].hist(post_crisis['cpi_mom'], bins=20, color='darkorange', edgecolor='black', alpha=0.7)
axes[1].set_title('Распределение ИПЦ: 2022–2026')
axes[1].set_xlabel('ИПЦ, %')
axes[1].set_ylabel('Частота')
axes[1].axvline(post_crisis['cpi_mom'].mean(), color='red', linestyle='--', label=f'Среднее: {post_crisis["cpi_mom"].mean():.2f}')
axes[1].legend()
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Статистическое сопоставление периодов

In [ ]:
def compute_stats(series, name):
    return {
        'Период': name,
        'Наблюдений': len(series),
        'Среднее': round(series.mean(), 4),
        'Медиана': round(series.median(), 4),
        'Ст. отклонение': round(series.std(ddof=1), 4),
        'Минимум': round(series.min(), 2),
        'Максимум': round(series.max(), 2),
        'Асимметрия': round(stats.skew(series), 4),
        'Эксцесс': round(stats.kurtosis(series, fisher=True), 4),
    }

stats_pre = compute_stats(pre_crisis['cpi_mom'], '1991–2021')
stats_post = compute_stats(post_crisis['cpi_mom'], '2022–2026')
stats_full = compute_stats(df['cpi_mom'], '1991–2026')

comparison = pd.DataFrame([stats_pre, stats_post, stats_full])
display(comparison)

## Проверка статистической значимости различий

In [ ]:
t_stat, p_value = stats.ttest_ind(pre_crisis['cpi_mom'], post_crisis['cpi_mom'], equal_var=False)
u_stat, u_p_value = stats.mannwhitneyu(pre_crisis['cpi_mom'], post_crisis['cpi_mom'], alternative='two-sided')

print(f"T-критерий Уэлча: t = {t_stat:.4f}, p-value = {p_value:.6f}")
print(f"Критерий Манна-Уитни: U = {u_stat:.0f}, p-value = {u_p_value:.6f}")

if p_value < 0.05:
    print("Различия между эпохами статистически значимы")
else:
    print("Различия между эпохами статистически незначимы")

## Тесты на равенство дисперсий и распределений

In [ ]:
levene_stat, levene_p = stats.levene(pre_crisis['cpi_mom'], post_crisis['cpi_mom'])
print(f"Тест Левена на равенство дисперсий: stat = {levene_stat:.4f}, p-value = {levene_p:.6f}")

if levene_p < 0.05:
    print("Дисперсии эпох статистически значимо различаются")
else:
    print("Дисперсии эпох статистически не различаются")

ks_stat, ks_p = stats.ks_2samp(pre_crisis['cpi_mom'], post_crisis['cpi_mom'])
print(f"Двухвыборочный тест Колмогорова-Смирнова: stat = {ks_stat:.4f}, p-value = {ks_p:.6f}")

## Анализ шоковых периодов по годам

In [ ]:
df['year'] = df['Date'].dt.year
yearly_stats = df.groupby('year')['cpi_mom'].agg(['mean', 'std', 'min', 'max']).round(2)
yearly_stats.columns = ['Среднее ИПЦ', 'Ст. откл.', 'Минимум', 'Максимум']
display(yearly_stats.tail(15))

## Визуализация средних значений по годам

In [ ]:
se = yearly_stats['Ст. откл.'] / np.sqrt(12)
ci = stats.t.ppf(0.975, 11) * se

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(yearly_stats.index, yearly_stats['Среднее ИПЦ'], marker='o', color='steelblue', linewidth=1.5)
ax.fill_between(yearly_stats.index, yearly_stats['Среднее ИПЦ'] - ci,
                yearly_stats['Среднее ИПЦ'] + ci, alpha=0.2, color='steelblue', label='95% ДИ среднего')
ax.axhline(100, color='red', linestyle='--', linewidth=0.8)
ax.axvline(2022, color='darkred', linestyle=':', linewidth=1.2, alpha=0.7, label='Структурный сдвиг')
ax.set_title('Средний месячный ИПЦ по годам с 95% доверительным интервалом')
ax.set_xlabel('Год')
ax.set_ylabel('ИПЦ, %')
ax.set_xticks(yearly_stats.index)
ax.set_xticklabels(yearly_stats.index, rotation=45, ha='right')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## Детальный анализ 2022 года

In [ ]:
year_2022 = df[df['year'] == 2022].copy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(year_2022['Date'].dt.month, year_2022['cpi_mom'] - 100, color='darkred', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Отклонение ИПЦ от 100% по месяцам 2022 года')
ax.set_xlabel('Месяц')
ax.set_ylabel('Отклонение, п.п.')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек'])
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

peak_month = year_2022.loc[year_2022['cpi_mom'].idxmax()]
print(f"Месяц 2022 года с наибольшим ИПЦ: {peak_month['Date'].strftime('%B %Y')}")
print(f"Значение ИПЦ: {peak_month['cpi_mom']:.2f}%")

## Автокорреляционный анализ полного ряда

In [ ]:
cpi_deviation = df['cpi_mom'] - 100

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

acf_vals = acf(cpi_deviation, nlags=24, fft=True)
pacf_vals = pacf(cpi_deviation, nlags=24)

axes[0].bar(range(len(acf_vals)), acf_vals, color='steelblue', alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.8)
confidence = 1.96 / np.sqrt(len(df))
axes[0].axhline(confidence, color='red', linestyle='--', linewidth=0.8, label='95% доверительный интервал')
axes[0].axhline(-confidence, color='red', linestyle='--', linewidth=0.8)
axes[0].set_title('Автокорреляционная функция ACF отклонений ИПЦ')
axes[0].set_xlabel('Лаг (месяцев)')
axes[0].set_ylabel('Коэффициент автокорреляции')
axes[0].legend()
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

axes[1].bar(range(len(pacf_vals)), pacf_vals, color='darkorange', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].axhline(confidence, color='red', linestyle='--', linewidth=0.8)
axes[1].axhline(-confidence, color='red', linestyle='--', linewidth=0.8)
axes[1].set_title('Частная автокорреляционная функция PACF отклонений ИПЦ')
axes[1].set_xlabel('Запаздывание месяцах')
axes[1].set_ylabel('Коэффициент частной автокорреляции')
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## Тест Дики-Фуллера на стационарность

In [ ]:
adf_result = adfuller(df['cpi_mom'], autolag='AIC', result_object=False)

print("Расширенный тест Дики-Фуллера для полного ряда:")
print(f"ADF статистика: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.6f}")
print(f"Количество лагов: {adf_result[2]}")
print(f"Критические значения:")
for key, value in adf_result[4].items():
    print(f"{key}: {value:.4f}")

if adf_result[1] < 0.05:
    print("Ряд стационарен по среднему")
else:
    print("Ряд нестационарен")

## Тест стационарности по периодам

In [ ]:
adf_pre = adfuller(pre_crisis['cpi_mom'], autolag='AIC', result_object=False)
adf_post = adfuller(post_crisis['cpi_mom'], autolag='AIC', result_object=False)

print(f"Тест Дики-Фуллера для 1991–2021: ADF = {adf_pre[0]:.4f}, p-value = {adf_pre[1]:.6f}")
print(f"Тест Дики-Фуллера для 2022–2026: ADF = {adf_post[0]:.4f}, p-value = {adf_post[1]:.6f}")

## Кумулятивный рост цен по десятилетиям

In [ ]:
df['decade'] = (df['Date'].dt.year // 10) * 10
decade_growth = df.groupby('decade')['cpi_mom'].apply(
    lambda x: (x / 100).prod()
).reset_index(name='cumulative_growth')
decade_growth['cumulative_growth'] = decade_growth['cumulative_growth'].round(2)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(decade_growth['decade'].astype(str), decade_growth['cumulative_growth'],
              color=['steelblue', 'steelblue', 'steelblue', 'darkorange'], edgecolor='black', alpha=0.8)
ax.set_title('Кумулятивный рост цен по десятилетиям')
ax.set_xlabel('Десятилетие')
ax.set_ylabel('Рост цен (раз)')
ax.set_yscale('log')
ax.grid(axis='y', linestyle='--', alpha=0.4)

for bar, val in zip(bars, decade_growth['cumulative_growth']):
    formatted_val = f"{val:,.0f}".replace(',', ' ') if val > 100 else f"{val:.1f}"
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f'x{formatted_val}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("Кумулятивный рост цен по десятилетиям:")
display(decade_growth)
print("Примечание: данные за 2020-е включают период только до сентября 2026 г.")

## Общий кумулятивный рост за весь период

In [ ]:
total_growth = (df['cpi_mom'] / 100).prod()
formatted_growth = f"{total_growth:,.0f}".replace(',', ' ')
print(f"Общий кумулятивный рост цен с января 1991 по сентябрь 2026: x{formatted_growth}")